# TextLens: Sentiment Analysis of FIFA World Cup 2022 Tweets
# TextLens: FIFA vb 2022 tweetek szentiment elemzése

This notebook guides you through a sentiment analysis task using tweets from the first day of the FIFA World Cup 2022. Sentiment analysis (opinion mining) is an NLP technique that decides whether a text is positive or negative. We cover data loading, text cleaning, feature extraction, model building and evaluation.

***

Ez a jegyzetfüzet egy szentiment elemzési feladaton vezet végig a FIFA vb 2022 első napjának tweetjei segítségével. A szentiment elemzés (véleménybányászat) olyan NLP technika, amely eldönti egy szövegről, hogy pozitív vagy negatív hangvételű-e. Foglalkozunk az adatok betöltésével, szövegtisztítással, jellemzők kinyerésével, modellépítéssel és értékeléssel.

## I need some data! / Kell némi adat!

- Dataset / Adathalmaz: [Tirendaz/fifa-world-cup-2022-tweets](https://huggingface.co/datasets/Tirendaz/fifa-world-cup-2022-tweets) (Hugging Face)
- ~30 000 tweets with the hashtag #WorldCup2022 / kb. 30 000 tweet a #WorldCup2022 hashtaggel
- Columns / Oszlopok: `Date Created`, `Number of Likes`, `Source of Tweet`, `Tweet`, `Sentiment`
- **Note / Megjegyzés:** the `Sentiment` labels were generated by a pre-trained model (cardiffnlp RoBERTa), not by humans. / A `Sentiment` címkéket egy előre betanított modell generálta, nem emberek.

## 1. Data Loading

We load the tweets into a pandas DataFrame. The original data has three classes (positive, neutral, negative). To keep a binary classification task (like a classic sentiment task), we keep only the **positive** and **negative** tweets. We also remove duplicate tweets (retweets), otherwise the same tweet could end up in both the training and the test set.

***

## 1. Adatok Betöltése

A tweeteket egy pandas DataFrame-be töltjük. Az eredeti adat három osztályt tartalmaz (pozitív, semleges, negatív). Hogy bináris osztályozási feladatunk legyen, csak a **pozitív** és a **negatív** tweeteket tartjuk meg. A duplikált tweeteket (retweeteket) is töröljük, különben ugyanaz a tweet a tanító és a teszt halmazba is bekerülhetne.

In [1]:
import pandas as pd  # Data handling / Adatkezelés

# Load the dataset from Hugging Face / Az adathalmaz betöltése a Hugging Face-ről
df_raw = pd.read_csv("hf://datasets/Tirendaz/fifa-world-cup-2022-tweets/fifa_world_cup_2022_tweets.csv")
df_raw.head()

C:\Users\Pc3\PycharmProjects\textlens2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0,Date Created,Number of Likes,Source of Tweet,Tweet,Sentiment
0,0,2022-11-20 23:59:21+00:00,4,Twitter Web App,What are we drinking today @TucanTribe \n@MadB...,neutral
1,1,2022-11-20 23:59:01+00:00,3,Twitter for iPhone,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,positive
2,2,2022-11-20 23:58:41+00:00,1,Twitter for iPhone,Worth reading while watching #WorldCup2022 htt...,positive
3,3,2022-11-20 23:58:33+00:00,1,Twitter Web App,Golden Maknae shinning bright\n\nhttps://t.co/...,positive
4,4,2022-11-20 23:58:28+00:00,0,Twitter for Android,"If the BBC cares so much about human rights, h...",negative


### Exercise 1: Data Loading / 1. Feladat: Adatok Betöltése

**Task / Feladat:** Display the first 3 rows, the column data types and the class distribution. / Jelenítse meg az első 3 sort, az oszlopok adattípusait és az osztályok eloszlását.

In [2]:
display(df_raw.head(3))  # First 3 rows / Első 3 sor
print(df_raw.dtypes)  # Column data types / Oszlopok adattípusai
print(df_raw["Sentiment"].value_counts())  # Class distribution / Osztályeloszlás

,Unnamed: 0,Date Created,Number of Likes,Source of Tweet,Tweet,Sentiment
0,0,2022-11-20 23:59:21+00:00,4,Twitter Web App,What are we drinking today @TucanTribe \n@MadB...,neutral
1,1,2022-11-20 23:59:01+00:00,3,Twitter for iPhone,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,positive
2,2,2022-11-20 23:58:41+00:00,1,Twitter for iPhone,Worth reading while watching #WorldCup2022 htt...,positive


Unnamed: 0         int64
Date Created         str
Number of Likes    int64
Source of Tweet      str
Tweet                str
Sentiment            str
dtype: object
Sentiment
positive    8489
neutral     8251
negative    5784
Name: count, dtype: int64


In [3]:
# Keep positive and negative tweets only / Csak a pozitív és negatív tweeteket tartjuk meg
df = df_raw[df_raw["Sentiment"].isin(["positive", "negative"])][["Tweet", "Sentiment"]]
df = df.dropna().drop_duplicates(subset="Tweet").reset_index(drop=True)  # Drop empty and duplicate tweets / Üres és duplikált tweetek törlése
df = df.rename(columns={"Tweet": "text", "Sentiment": "sentiment"})
df["text_raw"] = df["text"]  # Keep the original text for comparison / Megőrizzük az eredeti szöveget az összehasonlításhoz

print("Tweets:", len(df))
print((df["sentiment"].value_counts(normalize=True) * 100).round(1))  # Class balance in % / Osztályarány %-ban

Tweets: 14172
sentiment
positive    59.4
negative    40.6
Name: proportion, dtype: float64


## 2. Text Preprocessing and Cleaning

Tweets are noisy: links, @mentions, hashtags, HTML entities and slang. We clean them step by step:

*   **Lowercasing**: `Goal` and `goal` become the same word.
*   **HTML Removal**: remove HTML entities and tags (e.g. `&amp;`).
*   **URL and @mention Removal**: links and usernames carry no sentiment.
*   **Contraction Expansion**: `don't` becomes `do not`. This is done *before* removing punctuation, otherwise the apostrophe is lost and `don't` turns into `dont`.
*   **Special Character Removal**: keep only letters (hashtags lose the `#`, the word stays).
*   **Whitespace Removal**: remove extra spaces.
*   **Stop Word Removal**: remove very common words (`the`, `is`, `a`). We **keep the negations** (`no`, `nor`, `not`), because they flip the sentiment (`not good`).
*   **Lemmatization**: reduce words to their base form (`running` becomes `run`).

***

## 2. Szöveg Előfeldolgozás és Tisztítás

A tweetek zajosak: linkek, @említések, hashtagek, HTML entitások és szleng. Lépésről lépésre tisztítjuk őket:

*   **Kisbetűsítés**: a `Goal` és a `goal` ugyanaz a szó lesz.
*   **HTML Eltávolítása**: HTML entitások és címkék törlése (pl. `&amp;`).
*   **URL és @említés Eltávolítása**: a linkek és felhasználónevek nem hordoznak szentimentet.
*   **Rövidítések Kibontása**: a `don't` `do not` lesz. Ezt az írásjelek törlése *előtt* tesszük, különben az aposztróf elvész és a `don't` `dont` lesz.
*   **Speciális Karakterek Eltávolítása**: csak a betűket tartjuk meg (a hashtag elveszti a `#`-ot, a szó marad).
*   **Szóközök Eltávolítása**: a felesleges szóközök törlése.
*   **Stop Szavak Eltávolítása**: a nagyon gyakori szavak (`the`, `is`, `a`) törlése. A **tagadó szavakat megtartjuk** (`no`, `nor`, `not`), mert megfordítják a jelentést (`not good`).
*   **Lemmatizálás**: a szavakat alapalakra hozzuk (`running` lesz `run`).

In [4]:
# Lower case / Kisbetűsítés
df["text"] = df["text"].str.lower()
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing @canadasocceren #worldcup2022 launch ...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching #worldcup2022 htt...
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright\n\nhttps://t.co/...
3,"If the BBC cares so much about human rights, h...","if the bbc cares so much about human rights, h..."
4,"And like, will the mexican fans be able to scr...","and like, will the mexican fans be able to scr..."


In [5]:
# Remove HTML / HTML eltávolítása
import html
import re

def remove_html(text):
    text = html.unescape(text)  # &amp; -> & / &amp; -> &
    return re.sub(r"<[^>]+>", " ", text)  # Remove tags / Címkék törlése

df["text"] = df["text"].apply(remove_html)
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing @canadasocceren #worldcup2022 launch ...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching #worldcup2022 htt...
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright\n\nhttps://t.co/...
3,"If the BBC cares so much about human rights, h...","if the bbc cares so much about human rights, h..."
4,"And like, will the mexican fans be able to scr...","and like, will the mexican fans be able to scr..."


In [6]:
# Remove URLs and @mentions / URL-ek és @említések eltávolítása
def remove_urls_mentions(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # Links / Linkek
    text = re.sub(r"@\w+", " ", text)  # Usernames / Felhasználónevek
    return text

df["text"] = df["text"].apply(remove_urls_mentions)
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing #worldcup2022 launch video. shows h...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching #worldcup2022
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright\n\n \n#jeonjungk...
3,"If the BBC cares so much about human rights, h...","if the bbc cares so much about human rights, h..."
4,"And like, will the mexican fans be able to scr...","and like, will the mexican fans be able to scr..."


In [7]:
# Expand contractions / Rövidítések kibontása
CONTRACTIONS = {  # Order matters: specific forms first / A sorrend számít: előbb a speciális alakok
    "won't": "will not", "can't": "can not", "n't": " not",
    "'re": " are", "'ve": " have", "'ll": " will", "'d": " would", "'m": " am", "'s": " is",
}

def expand_contractions(text):
    text = text.replace("’", "'")  # Curly apostrophe -> straight / Görbe aposztróf -> egyenes
    for short, full in CONTRACTIONS.items():
        text = text.replace(short, full)
    return text

df["text"] = df["text"].apply(expand_contractions)
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing #worldcup2022 launch video. shows h...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching #worldcup2022
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright\n\n \n#jeonjungk...
3,"If the BBC cares so much about human rights, h...","if the bbc cares so much about human rights, h..."
4,"And like, will the mexican fans be able to scr...","and like, will the mexican fans be able to scr..."


In [8]:
# Remove special characters / Speciális karakterek eltávolítása
def remove_special_characters(text):
    return re.sub(r"[^a-z\s]", " ", text)  # Keep letters and spaces only / Csak betűk és szóközök maradnak

df["text"] = df["text"].apply(remove_special_characters)
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing worldcup launch video shows h...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching worldcup
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright\n\n \n jeonjungk...
3,"If the BBC cares so much about human rights, h...",if the bbc cares so much about human rights h...
4,"And like, will the mexican fans be able to scr...",and like will the mexican fans be able to scr...


In [9]:
# Remove white spaces / Szóközök eltávolítása
def remove_extra_whitespace(text):
    return " ".join(text.split())  # Strip and collapse multiple spaces / Szélek levágása, többszörös szóközök egyesítése

df["text"] = df["text"].apply(remove_extra_whitespace)
df[["text_raw", "text"]].head()

,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing worldcup launch video shows how much t...
1,Worth reading while watching #WorldCup2022 htt...,worth reading while watching worldcup
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright jeonjungkook jun...
3,"If the BBC cares so much about human rights, h...",if the bbc cares so much about human rights ho...
4,"And like, will the mexican fans be able to scr...",and like will the mexican fans be able to scre...


In [10]:
# Remove stop words / Stop szavak eltávolítása
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")  # English stop word list / Angol stop szó lista

NEGATIONS = {"no", "nor", "not"}  # Keep these, they change the sentiment / Ezeket megtartjuk, megváltoztatják a szentimentet
stop_words = set(stopwords.words("english")) - NEGATIONS

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in stop_words)

df["text"] = df["text"].apply(remove_stopwords)
df[["text_raw", "text"]].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pc3\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amazing worldcup launch video shows much face ...
1,Worth reading while watching #WorldCup2022 htt...,worth reading watching worldcup
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shinning bright jeonjungkook jun...
3,"If the BBC cares so much about human rights, h...",bbc cares much human rights homosexual rights ...
4,"And like, will the mexican fans be able to scr...",like mexican fans able scream puto homophobic ...


In [11]:
# Lemmatization / Lemmatizálás
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")  # WordNet dictionary / WordNet szótár
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    # Verb form first (running -> run), then noun form (goals -> goal)
    # Először igei alak (running -> run), aztán főnévi alak (goals -> goal)
    return " ".join(lemmatizer.lemmatize(lemmatizer.lemmatize(w, pos="v")) for w in text.split())

df["text"] = df["text"].apply(lemmatize_text)

# Remove tweets that became empty (e.g. only a link) / Üressé vált tweetek törlése (pl. csak link volt)
df = df[df["text"].str.len() > 0].reset_index(drop=True)
df[["text_raw", "text"]].head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Pc3\AppData\Roaming\nltk_data...


,text_raw,text
0,Amazing @CanadaSoccerEN #WorldCup2022 launch ...,amaze worldcup launch video show much face can...
1,Worth reading while watching #WorldCup2022 htt...,worth read watch worldcup
2,Golden Maknae shinning bright\n\nhttps://t.co/...,golden maknae shin bright jeonjungkook jungkoo...
3,"If the BBC cares so much about human rights, h...",bbc care much human right homosexual right wom...
4,"And like, will the mexican fans be able to scr...",like mexican fan able scream puto homophobic q...


### Exercise 2: Text Preprocessing / 2. Feladat: Szöveg Előfeldolgozás

**Task / Feladat:** Compare the average word count per tweet before and after cleaning. / Hasonlítsa össze az egy tweetre jutó átlagos szószámot a tisztítás előtt és után.

In [12]:
before = df["text_raw"].str.split().str.len().mean()  # Average words before cleaning / Átlagos szószám tisztítás előtt
after = df["text"].str.split().str.len().mean()  # Average words after cleaning / Átlagos szószám tisztítás után
print(f"Before cleaning: {before:.1f} words / tweet")
print(f"After cleaning:  {after:.1f} words / tweet")

Before cleaning: 19.2 words / tweet
After cleaning:  11.1 words / tweet


## 3. Data Preparation for Modeling

*   **Labeling**: convert `positive` / `negative` into 1 / 0.
*   **Train/Validation/Test Split** (80% / 10% / 10%): the training set is used to train the model, the validation set to watch for overfitting during training, the test set for the final, unbiased evaluation. We use `stratify`, so every subset keeps the same positive/negative ratio as the original data.

***

## 3. Adatok Előkészítése Modellezéshez

*   **Címkézés**: a `positive` / `negative` címkéket 1 / 0 értékre alakítjuk.
*   **Tanító/Validációs/Teszt Felosztás** (80% / 10% / 10%): a tanító halmazon tanul a modell, a validációs halmazon figyeljük a túltanulást, a teszt halmazon mérjük a végső, elfogulatlan teljesítményt. A `stratify` paraméterrel minden részhalmazban megmarad az eredeti pozitív/negatív arány.

In [13]:
# Labeling / Címkézés
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})  # positive -> 1, negative -> 0 / pozitív -> 1, negatív -> 0
df[["text", "sentiment", "label"]].head()

,text,sentiment,label
0,amaze worldcup launch video show much face can...,positive,1
1,worth read watch worldcup,positive,1
2,golden maknae shin bright jeonjungkook jungkoo...,positive,1
3,bbc care much human right homosexual right wom...,negative,0
4,like mexican fan able scream puto homophobic q...,negative,0


In [14]:
# Train, validation and test set / Tanító, validációs és teszt halmaz
from sklearn.model_selection import train_test_split

# 80% train, 20% temporary / 80% tanító, 20% ideiglenes
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
# Temporary -> 50% validation, 50% test / Ideiglenes -> 50% validációs, 50% teszt
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Labels / Címkék
train_labels = train_df["label"].values
val_labels = val_df["label"].values
test_labels = test_df["label"].values

Train set size: 11337
Validation set size: 1417
Test set size: 1418


### Exercise: Data Preparation / Feladat: Adatok Előkészítése

**Task / Feladat:** Show the percentage of positive and negative tweets in the original data and in each split. / Jelenítse meg a pozitív és negatív tweetek százalékos arányát az eredeti adatban és minden részhalmazban.

In [15]:
for name, part in [("original", df), ("train", train_df), ("val", val_df), ("test", test_df)]:
    ratio = (part["label"].value_counts(normalize=True) * 100).round(1)
    print(f"{name:9s} negative: {ratio[0]}%  positive: {ratio[1]}%")

original  negative: 40.6%  positive: 59.4%
train     negative: 40.5%  positive: 59.5%
val       negative: 40.6%  positive: 59.4%
test      negative: 40.6%  positive: 59.4%


## 4. Feature Extraction

Machine learning models cannot process raw text, only numbers. We try two techniques:

*   **CountVectorizer**: counts how many times each word appears in a tweet.
*   **TF-IDF Vectorizer**: weights each word by its frequency in the tweet (TF) and by how rare it is across all tweets (IDF). Words that appear everywhere get a low weight.

The vectorizers are **fitted on the training set only**, and then used to transform the validation and test sets. Otherwise information from the test set would leak into the training.

***

## 4. Jellemzők Kinyerése

A gépi tanulási modellek nem tudnak nyers szöveget feldolgozni, csak számokat. Két technikát próbálunk ki:

*   **CountVectorizer**: megszámolja, hányszor szerepel az egyes szó egy tweetben.
*   **TF-IDF Vectorizer**: a szavakat a tweeten belüli gyakoriságuk (TF) és az összes tweetben mért ritkaságuk (IDF) alapján súlyozza. Az mindenhol előforduló szavak alacsony súlyt kapnak.

A vektorizálókat **csak a tanító halmazra illesztjük**, majd ezzel alakítjuk át a validációs és teszt halmazt. Különben a teszt halmaz információja beszivárogna a tanításba.

In [16]:
# CountVectorizer / CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1000)  # Top 1000 most frequent words / Az 1000 leggyakoribb szó

X_train_cv = vectorizer.fit_transform(train_df["text"]).toarray()  # Fit on train + transform / Illesztés a tanítón + átalakítás
X_val_cv = vectorizer.transform(val_df["text"]).toarray()  # Transform only / Csak átalakítás
X_test_cv = vectorizer.transform(test_df["text"]).toarray()  # Transform only / Csak átalakítás

print("CountVectorizer train shape:", X_train_cv.shape)

CountVectorizer train shape: (11337, 1000)


In [17]:
# TF-IDF / TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=1000)  # Top 1000 words / Az 1000 leggyakoribb szó

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["text"]).toarray()
X_val_tfidf = tfidf_vectorizer.transform(val_df["text"]).toarray()
X_test_tfidf = tfidf_vectorizer.transform(test_df["text"]).toarray()

print("TF-IDF train shape:", X_train_tfidf.shape)

TF-IDF train shape: (11337, 1000)


### Exercise: Feature Extraction / Feladat: Jellemzők Kinyerése

**Task / Feladat:** Create new vectorizers with `max_features=500` and `ngram_range=(1, 2)` (bigrams), apply them to the training texts and print the shapes. / Hozzon létre új vektorizálókat `max_features=500` és `ngram_range=(1, 2)` (bigramok) beállítással, alkalmazza őket a tanító szövegekre és írja ki az alakjukat.

In [18]:
cv_bigram = CountVectorizer(max_features=500, ngram_range=(1, 2))
tfidf_bigram = TfidfVectorizer(max_features=500, ngram_range=(1, 2))

print("CountVectorizer (1-2 gram):", cv_bigram.fit_transform(train_df["text"]).shape)
print("TF-IDF (1-2 gram):", tfidf_bigram.fit_transform(train_df["text"]).shape)
print("Example bigrams / Példa bigramok:", [w for w in cv_bigram.get_feature_names_out() if " " in w][:10])

CountVectorizer (1-2 gram): (11337, 500)
TF-IDF (1-2 gram): (11337, 500)
Example bigrams / Példa bigramok: ['already worldcup', 'begin worldcup', 'ceremony worldcup', 'could not', 'cup begin', 'cup day', 'cup market', 'cup open', 'cup qatar', 'cup start']


## 5. Model Building and Training

We build simple neural networks with TensorFlow/Keras. All of them are binary classifiers with a **sigmoid** output: a value close to 1 means positive, close to 0 means negative.

*   **model_cv**: a single sigmoid neuron on the CountVectorizer features.
*   **model_tfidf**: a single sigmoid neuron on the TF-IDF features.
*   **model_cv_hidden**: like `model_cv`, but with an extra hidden `Dense` layer (10 neurons, ReLU), which can learn non-linear patterns.

***

## 5. Modell Építés és Tanítás

Egyszerű neurális hálókat építünk TensorFlow/Keras segítségével. Mindegyik bináris osztályozó **sigmoid** kimenettel: az 1-hez közeli érték pozitív, a 0-hoz közeli negatív véleményt jelent.

*   **model_cv**: egyetlen sigmoid neuron a CountVectorizer jellemzőkön.
*   **model_tfidf**: egyetlen sigmoid neuron a TF-IDF jellemzőkön.
*   **model_cv_hidden**: mint a `model_cv`, de egy extra rejtett `Dense` réteggel (10 neuron, ReLU), amely nem-lineáris mintákat is meg tud tanulni.

In [19]:
# Model CV / Modell CountVectorizerrel
import tensorflow as tf

tf.keras.utils.set_random_seed(42)  # Reproducible results / Reprodukálható eredmények

model_cv = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_cv.shape[1],)),  # 1000 input features / 1000 bemeneti jellemző
    tf.keras.layers.Dense(1, activation="sigmoid"),  # Binary output / Bináris kimenet
])
model_cv.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_cv = model_cv.fit(X_train_cv, train_labels, epochs=5, batch_size=50,
                          validation_data=(X_val_cv, val_labels), verbose=2)

Epoch 1/5
227/227 - 1s - 4ms/step - accuracy: 0.6886 - loss: 0.6304 - val_accuracy: 0.7615 - val_loss: 0.5695
Epoch 2/5
227/227 - 0s - 1ms/step - accuracy: 0.7902 - loss: 0.5412 - val_accuracy: 0.8066 - val_loss: 0.5038
Epoch 3/5
227/227 - 0s - 1ms/step - accuracy: 0.8239 - loss: 0.4870 - val_accuracy: 0.8292 - val_loss: 0.4607
Epoch 4/5
227/227 - 0s - 1ms/step - accuracy: 0.8403 - loss: 0.4497 - val_accuracy: 0.8412 - val_loss: 0.4304
Epoch 5/5
227/227 - 0s - 1ms/step - accuracy: 0.8507 - loss: 0.4223 - val_accuracy: 0.8490 - val_loss: 0.4079


In [20]:
# Model TFIDF / Modell TF-IDF-fel
model_tfidf = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_tfidf.shape[1],)),  # Uses the TF-IDF shape / A TF-IDF alakját használja
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model_tfidf.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_tfidf = model_tfidf.fit(X_train_tfidf, train_labels, epochs=5, batch_size=50,
                                validation_data=(X_val_tfidf, val_labels), verbose=2)

Epoch 1/5
227/227 - 1s - 3ms/step - accuracy: 0.6138 - loss: 0.6648 - val_accuracy: 0.6288 - val_loss: 0.6380
Epoch 2/5
227/227 - 0s - 1ms/step - accuracy: 0.6549 - loss: 0.6182 - val_accuracy: 0.6845 - val_loss: 0.5980
Epoch 3/5
227/227 - 0s - 1ms/step - accuracy: 0.7119 - loss: 0.5825 - val_accuracy: 0.7347 - val_loss: 0.5654
Epoch 4/5
227/227 - 0s - 1ms/step - accuracy: 0.7634 - loss: 0.5523 - val_accuracy: 0.7812 - val_loss: 0.5374
Epoch 5/5
227/227 - 0s - 1ms/step - accuracy: 0.8009 - loss: 0.5261 - val_accuracy: 0.8095 - val_loss: 0.5129


### Exercise: Model Building and Training / Feladat: Modell Építés és Tanítás

**Task / Feladat:** Add a hidden `Dense` layer (10 units, ReLU) before the output layer of `model_cv`, retrain it and observe the validation accuracy. / Adjon egy rejtett `Dense` réteget (10 egység, ReLU) a `model_cv` kimeneti rétege elé, tanítsa újra és figyelje meg a validációs pontosságot.

In [21]:
# Model CV with hidden layer / Modell CountVectorizerrel és rejtett réteggel
model_cv_hidden = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train_cv.shape[1],)),
    tf.keras.layers.Dense(10, activation="relu"),  # Hidden layer / Rejtett réteg
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model_cv_hidden.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history_cv_hidden = model_cv_hidden.fit(X_train_cv, train_labels, epochs=5, batch_size=50,
                                        validation_data=(X_val_cv, val_labels), verbose=2)

print("Last validation accuracy / Utolsó validációs pontosság:")
print(f"  model_cv:        {history_cv.history['val_accuracy'][-1]:.4f}")
print(f"  model_tfidf:     {history_tfidf.history['val_accuracy'][-1]:.4f}")
print(f"  model_cv_hidden: {history_cv_hidden.history['val_accuracy'][-1]:.4f}")

Epoch 1/5
227/227 - 1s - 4ms/step - accuracy: 0.7089 - loss: 0.5449 - val_accuracy: 0.8454 - val_loss: 0.4219
Epoch 2/5
227/227 - 0s - 1ms/step - accuracy: 0.8618 - loss: 0.3692 - val_accuracy: 0.8694 - val_loss: 0.3301
Epoch 3/5
227/227 - 0s - 1ms/step - accuracy: 0.8822 - loss: 0.3041 - val_accuracy: 0.8800 - val_loss: 0.3004
Epoch 4/5
227/227 - 0s - 1ms/step - accuracy: 0.8944 - loss: 0.2741 - val_accuracy: 0.8814 - val_loss: 0.2902
Epoch 5/5
227/227 - 0s - 1ms/step - accuracy: 0.8994 - loss: 0.2567 - val_accuracy: 0.8843 - val_loss: 0.2874
Last validation accuracy / Utolsó validációs pontosság:
  model_cv:        0.8490
  model_tfidf:     0.8095
  model_cv_hidden: 0.8843


## 6. Model Evaluation

We evaluate the trained models on the **test set**, which they have never seen. Accuracy alone can be misleading when the classes are not balanced (a model that always says "positive" could still look good), so we also look at precision, recall and the F1-score for both classes.

***

## 6. Modell Értékelés

A betanított modelleket a **teszt halmazon** értékeljük, amelyet a modell soha nem látott. A pontosság önmagában félrevezető lehet, ha az osztályok nem kiegyensúlyozottak (egy modell, amely mindig "pozitívat" mond, is jónak tűnhet), ezért a precíziót, a visszahívást (recall) és az F1-pontszámot is megnézzük mindkét osztályra.

In [22]:
# Test accuracy / Teszt pontosság
loss_cv, acc_cv = model_cv.evaluate(X_test_cv, test_labels, verbose=0)
loss_tfidf, acc_tfidf = model_tfidf.evaluate(X_test_tfidf, test_labels, verbose=0)
loss_hidden, acc_hidden = model_cv_hidden.evaluate(X_test_cv, test_labels, verbose=0)

print(f"Test accuracy (CV):        {acc_cv:.4f}")
print(f"Test accuracy (TF-IDF):    {acc_tfidf:.4f}")
print(f"Test accuracy (CV hidden): {acc_hidden:.4f}")

Test accuracy (CV):        0.8477
Test accuracy (TF-IDF):    0.8047
Test accuracy (CV hidden): 0.8752


### Exercise: Model Evaluation / Feladat: Modell Értékelés

**Task / Feladat:** Calculate precision, recall and F1-score for the models on the test set. / Számolja ki a precíziót, a visszahívást és az F1-pontszámot a modellekre a teszt halmazon.

In [23]:
from sklearn.metrics import classification_report

def report(model, X_test, name):
    predictions = (model.predict(X_test, verbose=0) > 0.5).astype(int).ravel()  # Probability -> 0/1 / Valószínűség -> 0/1
    print(f"=== {name} ===")
    print(classification_report(test_labels, predictions, target_names=["negative", "positive"], digits=3))

report(model_cv, X_test_cv, "model_cv")
report(model_tfidf, X_test_tfidf, "model_tfidf")
report(model_cv_hidden, X_test_cv, "model_cv_hidden")

=== model_cv ===
              precision    recall  f1-score   support

    negative      0.878     0.725     0.794       575
    positive      0.832     0.931     0.879       843

    accuracy                          0.848      1418
   macro avg      0.855     0.828     0.837      1418
weighted avg      0.851     0.848     0.845      1418

=== model_tfidf ===
              precision    recall  f1-score   support

    negative      0.921     0.567     0.702       575
    positive      0.766     0.967     0.855       843

    accuracy                          0.805      1418
   macro avg      0.843     0.767     0.778      1418
weighted avg      0.829     0.805     0.793      1418

=== model_cv_hidden ===
              precision    recall  f1-score   support

    negative      0.849     0.842     0.845       575
    positive      0.893     0.898     0.895       843

    accuracy                          0.875      1418
   macro avg      0.871     0.870     0.870      1418
weighted avg 

# Extra

## Try it yourself / Próbáld ki magad

We put the whole cleaning pipeline into one function, so we can classify any new tweet.

***

A teljes tisztítási folyamatot egyetlen függvénybe tesszük, így bármilyen új tweetet osztályozhatunk.

In [ ]:
def clean_text(text):
    text = remove_html(text.lower())
    text = remove_urls_mentions(text)
    text = expand_contractions(text)
    text = remove_special_characters(text)
    text = remove_extra_whitespace(text)
    text = remove_stopwords(text)
    return lemmatize_text(text)

def predict_sentiment(tweet):
    features = tfidf_vectorizer.transform([clean_text(tweet)]).toarray()
    score = float(model_tfidf.predict(features, verbose=0)[0][0])
    return ("positive" if score > 0.5 else "negative"), round(score, 3)

for tweet in ["What a goal! Absolutely brilliant opening ceremony #WorldCup2022",
              "This is a disgrace, the boycott is justified and the organisation is terrible",
              "Not a good game at all, so boring"]:
    print(predict_sentiment(tweet), "<-", tweet)

## Conclusion

*   Cleaning matters a lot for tweets: links, mentions and hashtags are noise, while negations (`not`) must be kept.
*   A single sigmoid neuron is mathematically the same as **logistic regression** on the word features. The hidden layer adds non-linearity, but on 1000 word-count features the gain can be small.
*   Accuracy must be read together with precision, recall and F1, because the positive and negative classes are not perfectly balanced.
*   **Limitations:** the labels come from another model (not humans), neutral tweets and emojis were left out, and sarcasm is hard for bag-of-words models.
*   **Next steps:** keep the neutral class (3-class softmax), use bigrams, handle emojis.

***

## Konklúzió

*   A tisztítás sokat számít a tweeteknél: a linkek, említések és hashtagek zajnak számítanak, a tagadó szavakat (`not`) viszont meg kell tartani.
*   Egyetlen sigmoid neuron matematikailag ugyanaz, mint a **logisztikus regresszió** a szójellemzőkön. A rejtett réteg nem-linearitást ad, de 1000 szószám-jellemzőn a nyereség kicsi lehet.
*   A pontosságot a precízióval, a visszahívással és az F1-gyel együtt kell olvasni, mert a pozitív és negatív osztály nem teljesen kiegyensúlyozott.
*   **Korlátok:** a címkéket egy másik modell adta (nem emberek), a semleges tweetek és az emojik kimaradtak, a szarkazmust pedig nehezen kezelik a szózsák modellek.
*   **Következő lépések:** a semleges osztály megtartása (3 osztályos softmax), bigramok használata, emojik kezelése.